# E4 — Seven-class gas identification

Compare linear, tree, random-kernel, convolutional, and attention-based classifiers on single-gas identification. The chronological test uses the subsequent acquisition day; random partitions are descriptive controls, not future-operation estimates.

In [ ]:
from pathlib import Path
import os, re, sys, json, hashlib, time, random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix

P = Path.cwd().resolve()
SRC = next((q for p in (P, *P.parents) for q in (p, p / 'src') if (q / 'esdata').is_dir()), None)
if SRC is None:
    raise FileNotFoundError('Run from the repository root or src/')
sys.path.insert(0, str(SRC))
from esdata import EsmellDataset, Chain
from esdata.transforms import (DefaultCSVPreprocessing, SimplifyLabels, SelectRChannels,
    UserLabelCycleSegments, CoolingHeatingSegments, CanonicalTempAlign,
    CanonicalSubtractionFeatures)

DATA_DIR = Path(os.getenv('ESMELL_DATA_DIR', str(SRC.parent / 'data' / 'raw')))
OUT = SRC / 'outputs' / 'E4_single_gas'
OUT.mkdir(parents=True, exist_ok=True)
ARCHIVES = {171: 'curiosity-5v16.171.0.tar.gz', 172: 'curiosity-1v17.172.0.tar.gz'}
CLASSES = ('air', 'acetone', 'ammonia', 'benzene', 'isopropanol', 'pentane', 'toluene')
METHODS = ('LogReg64', 'ExtraTrees64', 'ExtraTrees800', 'ROCKET', 'CNN_E2', 'ConvTran')
DEEP = ('CNN_E2', 'ConvTran')
CHRONO_SEEDS = (42, 43, 44)
RANDOM_SPLITS = (11, 22, 33)
RANDOM_DEEP_SEED = 42
EPOCHS, PATIENCE, BATCH = 60, 12, 32
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for fname in ARCHIVES.values():
    if not (DATA_DIR / fname).is_file():
        raise FileNotFoundError(DATA_DIR / fname)
print('Device:', DEVICE, '| Output:', OUT)

## 1. Records and splits

Use paired campaigns 171/172 as separate device datasets, with the original air-referenced heating/hysteresis features. Apply identical temporal boundaries and class vocabularies.

In [ ]:
def label_id(text):
    s = re.sub(r'\[[^]]*\]', '', str(text).lower()).split('_d', 1)[0].strip()
    s = {'none': 'air', 'amonia': 'ammonia'}.get(s, s)
    if s not in CLASSES:
        raise ValueError(f'Unexpected class: {text!r}')
    return CLASSES.index(s)


def load_card(card):
    source = EsmellDataset(root_path=DATA_DIR, subfolders=ARCHIVES[card],
                           csv_repair='strict', archive_cache='disk', preload=True)
    pipe = source >> Chain(DefaultCSVPreprocessing(), SimplifyLabels(),
        SelectRChannels(include=('R1', 'R2')),
        UserLabelCycleSegments(),
        CoolingHeatingSegments(temp_col='T_mean', smooth_window=3, eps=.2),
        CanonicalTempAlign(n_grid=200, temp_range=(175., 390.),
                           normalize=False, conductance=True),
        CanonicalSubtractionFeatures(features=((1, -3, 5), ((1, -3, 5), (0, -2, 4)))))
    rows, curves = [], []
    for sid in source.ids:
        info = source.metadata(sid)
        stamp = pd.Timestamp(info['datetime'])
        if not (pd.Timestamp('2026-10-19') <= stamp < pd.Timestamp('2026-10-22')):
            continue
        x = np.asarray(pipe.features(sid), dtype=np.float32)
        if x.shape != (4, 200) or not np.isfinite(x).all():
            raise ValueError(f'{card} {sid}: invalid feature tensor {x.shape}')
        rows.append((str(sid), stamp, label_id(info['classname'])))
        curves.append(x)
    meta = pd.DataFrame(rows, columns=['sample_id', 'time', 'y'])
    if len(meta) != 635 or meta.sample_id.duplicated().any():
        raise AssertionError(f'{card}: expected exactly 635 unique recordings; found {len(meta)}')
    return meta, np.stack(curves)


def stats64(x):
    a = [x.mean(2), x.std(2), x.min(2), x.max(2),
         np.median(x, axis=2), np.quantile(np.abs(x), .9, axis=2)]
    a.extend(b.mean(2) for b in np.array_split(x, 10, axis=2))
    return np.concatenate(a, axis=1).astype(np.float32)

meta171, x171 = load_card(171)
meta172, x172 = load_card(172)
a = meta171.assign(i171=np.arange(len(meta171))).sort_values('time')
b = meta172.assign(i172=np.arange(len(meta172))).rename(
    columns={'time': 'time_172', 'sample_id': 'sample_172'}).sort_values('time_172')
# The dataset audit confirms a one-second clock offset on 13 of the paired exposures.
pairs = pd.merge_asof(a, b, left_on='time', right_on='time_172', by='y',
                      direction='nearest', tolerance=pd.Timedelta(seconds=2))
if pairs.i172.isna().any() or pairs.i172.duplicated().any():
    raise AssertionError('Incomplete or ambiguous 171/172 matched pair')
pairs = pairs.sort_values('time').reset_index(drop=True)
assert len(pairs) == 635 and pairs.time.is_unique
assert (pairs.time-pairs.time_172).abs().max() <= pd.Timedelta(seconds=1)
DATA = {171: x171[pairs.i171.to_numpy(dtype=int)], 172: x172[pairs.i172.to_numpy(dtype=int)]}
y = pairs.y.to_numpy(dtype=np.int64)
assert set(y) == set(range(7))
assert all(X.shape == (635, 4, 200) for X in DATA.values())
FEATURES = {card: stats64(X) for card, X in DATA.items()}
assert all(z.shape == (635, 64) for z in FEATURES.values())

# Chronological split is shared by both cards, and time boundaries are fixed.
t = pairs.time
chrono = (
    np.flatnonzero(t < pd.Timestamp('2026-10-20 12:00')),
    np.flatnonzero((t >= pd.Timestamp('2026-10-20 12:00')) & (t < pd.Timestamp('2026-10-21'))),
    np.flatnonzero(t >= pd.Timestamp('2026-10-21')),
)
assert tuple(map(len, chrono)) == (244, 134, 257)


def check_split(split):
    tr, va, te = map(np.asarray, split)
    assert len(set(tr) & set(va)) == len(set(tr) & set(te)) == len(set(va) & set(te)) == 0
    assert len(np.unique(np.r_[tr, va, te])) == 635
    assert all(set(y[part]) == set(range(7)) for part in (tr, va, te))


def random_split(seed):
    trval, te = train_test_split(np.arange(635), test_size=257,
                                stratify=y, random_state=seed)
    tr, va = train_test_split(trval, test_size=134,
                              stratify=y[trval], random_state=seed + 1000)
    return np.asarray(tr), np.asarray(va), np.asarray(te)

SPLITS = [('future', 0, chrono)] + [('random', k, random_split(k)) for k in RANDOM_SPLITS]
for _, _, split in SPLITS:
    check_split(split)

manifest = []
for protocol, repeat, split in SPLITS:
    for part, indices in zip(('train', 'validation', 'test'), split):
        for i in indices:
            manifest.append({'protocol': protocol, 'repeat': repeat, 'split': part,
                'time': pairs.time[i], 'pair_id': str(pairs.time[i]),
                'sample_171': pairs.sample_id[i], 'sample_172': pairs.sample_172[i],
                'class': CLASSES[y[i]]})
pd.DataFrame(manifest).to_csv(OUT / 'splits.csv', index=False)
print('Pairs:', len(pairs), '| Chronological train / val / test:', tuple(map(len, chrono)))
print('Classes in chronological split:\n', pd.DataFrame({
    name: np.bincount(y[idx], minlength=7) for name, idx in zip(('train','val','test'), chrono)
}, index=CLASSES))

## 2. Model families

Compare logistic regression, ExtraTrees (summary and full-curve features), ROCKET, 1D CNN, and a compact ConvTran adaptation. These are full model–representation configurations, not a controlled architecture-only comparison.

In [ ]:
class EsmellEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        def block(a, b, k):
            return nn.Sequential(nn.Conv1d(a,b,k,padding=k//2),nn.BatchNorm1d(b),nn.ReLU(),
                nn.Conv1d(b,b,k,padding=k//2),nn.BatchNorm1d(b),nn.ReLU(),nn.MaxPool1d(2))
        self.features = nn.Sequential(block(4,64,7),block(64,128,5))
        self.projection = nn.Linear(128,128)
    def forward(self, x):
        return self.projection(self.features(x).mean(-1))


class CNN_E2(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EsmellEncoder()
        self.classifier = nn.Sequential(nn.Linear(128,128),nn.ReLU(),nn.Dropout(.2),nn.Linear(128,7))
    def forward(self, x):
        return self.classifier(self.encoder(x))


class ConvTran(nn.Module):
    # Original ConvTran ingredients: temporal+spatial conv, tAPE and eRPE after softmax.
    def __init__(self, channels=4, length=200, emb=16, heads=8, ff=256, dropout=.1):
        super().__init__()
        self.emb, self.heads = emb, heads
        self.embed = nn.Sequential(nn.ZeroPad2d((3,4,0,0)),
            nn.Conv2d(1,emb*4,(1,8)),nn.BatchNorm2d(emb*4),nn.GELU(),
            nn.Conv2d(emb*4,emb,(channels,1)),nn.BatchNorm2d(emb),nn.GELU())
        pos = torch.arange(length).float()[:,None]
        div = torch.exp(torch.arange(0,emb,2).float()*(-np.log(10000.)/emb))
        pe = torch.zeros(length,emb)
        pe[:,0::2] = torch.sin(pos*div*(emb/length))
        pe[:,1::2] = torch.cos(pos*div*(emb/length))
        self.register_buffer('tape',pe[None])
        idx = torch.arange(length)
        self.register_buffer('rel_idx',idx[:,None]-idx[None,:]+length-1)
        self.rel = nn.Parameter(torch.zeros(2*length-1, heads))
        self.q, self.k, self.v = (nn.Linear(emb,emb,bias=False) for _ in range(3))
        self.attn_norm = nn.LayerNorm(emb)
        self.norm1, self.norm2 = nn.LayerNorm(emb), nn.LayerNorm(emb)
        self.drop = nn.Dropout(dropout)
        self.ff = nn.Sequential(nn.Linear(emb,ff),nn.ReLU(),nn.Dropout(dropout),
                                nn.Linear(ff,emb),nn.Dropout(dropout))
        self.head = nn.Linear(emb,7)
    def forward(self, x):
        z = self.embed(x[:,None]).squeeze(2).transpose(1,2)  # B,L,E
        p = self.drop(z+self.tape)
        b,l,e = p.shape
        def heads(m): return m.reshape(b,l,self.heads,e//self.heads).transpose(1,2)
        q,k,v = heads(self.q(p)), heads(self.k(p)), heads(self.v(p))
        attn = (q@k.transpose(-2,-1))*(e**-.5)
        attn = attn.softmax(-1) + self.rel[self.rel_idx].permute(2,0,1)[None]
        h = (attn@v).transpose(1,2).reshape(b,l,e)
        z = self.norm1(z+self.attn_norm(h))
        z = self.norm2(z+self.ff(z))
        return self.head(z.mean(1))


def make_network(name):
    return CNN_E2() if name == 'CNN_E2' else ConvTran()

# Shape check without fitting or reading test targets.
for name in DEEP:
    net = make_network(name)
    with torch.no_grad():
        assert net(torch.zeros(2,4,200)).shape == (2,7)
    print(name, 'parameters:', sum(p.numel() for p in net.parameters()))

In [ ]:
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def norm3d(X, fit):
    mu = X[fit].mean((0,2),keepdims=True)
    std = X[fit].std((0,2),keepdims=True)
    return ((X-mu)/np.maximum(std,1.e-6)).astype(np.float32)


def batches(X, y, indices, seed, shuffle):
    dataset = TensorDataset(torch.from_numpy(X[indices]), torch.from_numpy(y[indices]))
    return DataLoader(dataset, batch_size=BATCH, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(seed),num_workers=0)


def predict_net(net, X, indices):
    net.eval()
    with torch.no_grad():
        return np.concatenate([net(xb.to(DEVICE)).argmax(1).cpu().numpy()
          for xb,_ in batches(X,y,indices,42,False)])


def train_epochs(net, X, fit, n_epochs, seed, lr, history=None, val=None):
    count = np.bincount(y[fit],minlength=7)
    weights = torch.tensor(len(fit)/(7*np.maximum(count,1)),device=DEVICE,dtype=torch.float32)
    opt = torch.optim.AdamW(net.parameters(),lr=lr,weight_decay=1e-4)
    best_epoch, best_score, stale = 1, -1., 0
    for ep in range(1,n_epochs+1):
        net.train()
        for xb,yb in batches(X,y,fit,seed+ep,True):
            opt.zero_grad()
            loss = nn.functional.cross_entropy(net(xb.to(DEVICE)),yb.to(DEVICE),weight=weights)
            loss.backward(); opt.step()
        if val is not None:
            score = balanced_accuracy_score(y[val],predict_net(net,X,val))
            if history is not None: history.append({'epoch':ep,'val_bacc':score})
            if score > best_score+1e-9:
                best_epoch,best_score,stale=ep,score,0
            else:
                stale+=1
                if stale>=PATIENCE: break
    return best_epoch,best_score


def fit_deep(name, X, tr, va, te, seed, history):
    seed_all(seed)
    lr = 1e-3 if name=='CNN_E2' else 5e-4
    Xtr = norm3d(X,tr)
    net = make_network(name).to(DEVICE)
    track=[]
    epoch, val_score = train_epochs(net,Xtr,tr,EPOCHS,seed,lr,history=track,val=va)
    for item in track:
        history.append({'model':name,'seed':seed,**item})
    seed_all(seed)
    fit = np.sort(np.r_[tr,va])
    Xfit = norm3d(X,fit)
    final_net = make_network(name).to(DEVICE)
    train_epochs(final_net,Xfit,fit,epoch,seed,lr)
    return predict_net(final_net,Xfit,te), epoch, val_score, sum(p.numel() for p in final_net.parameters())


def fit_classic(name, X, Z, tr, va, te, seed):
    fit = np.sort(np.r_[tr,va])
    if name=='LogReg64':
        model = make_pipeline(StandardScaler(),LogisticRegression(
            C=1.,max_iter=3000,class_weight='balanced',random_state=seed))
        U = Z
    elif name.startswith('ExtraTrees'):
        model = ExtraTreesClassifier(n_estimators=300,min_samples_leaf=1,
                max_features='sqrt',class_weight='balanced',random_state=seed,n_jobs=-1)
        U = Z if name=='ExtraTrees64' else X.reshape(len(X),-1)
    elif name=='ROCKET':
        try:
            from sktime.classification.kernel_based import RocketClassifier
        except ImportError as e:
            raise ImportError('ROCKET requires: pip install sktime numba') from e
        model = RocketClassifier(num_kernels=10000,rocket_transform='rocket',
                                 use_multivariate='yes',random_state=seed,n_jobs=1)
        U = norm3d(X,fit)
    else:
        raise KeyError(name)
    model.fit(U[fit],y[fit])
    return model.predict(U[te]).astype(np.int64)

## 3. Fixed benchmark

Fit transforms on training data. Select neural training duration on validation, then refit on the full pretest cohort. No method is selected using the chronological test.

In [ ]:
scores, predictions, histories, recalls, cms = [], [], [], [], []
for card in (171, 172):
    X, Z = DATA[card], FEATURES[card]
    for protocol, repeat, (tr,va,te) in SPLITS:
        for name in METHODS:
            seeds = (CHRONO_SEEDS if protocol=='future' else (RANDOM_DEEP_SEED,)) if name in DEEP else (42,)
            for seed in seeds:
                t0 = time.perf_counter()
                h=[]
                if name in DEEP:
                    pred, epoch, val_bacc, nparams = fit_deep(name,X,tr,va,te,seed,h)
                else:
                    pred = fit_classic(name,X,Z,tr,va,te,seed)
                    epoch, val_bacc, nparams = None, np.nan, None
                elapsed = time.perf_counter()-t0
                row = {'card':card,'protocol':protocol,'repeat':repeat,'model':name,'seed':seed,
                       'bacc':balanced_accuracy_score(y[te],pred),
                       'macro_f1':f1_score(y[te],pred,average='macro',labels=range(7),zero_division=0),
                       'fit_seconds':elapsed,'best_epoch':epoch,'val_bacc':val_bacc,'parameters':nparams}
                scores.append(row)
                for i,p in zip(te,pred):
                    predictions.append({'card':card,'protocol':protocol,'repeat':repeat,'model':name,
                        'seed':seed,'pair_id':str(pairs.time[i]),'sample_id':pairs.sample_id[i] if card==171 else pairs.sample_172[i],
                        'y_true':CLASSES[y[i]],'y_pred':CLASSES[p]})
                cm = confusion_matrix(y[te],pred,labels=range(7))
                for k in range(7):
                    for j in range(7):
                        cms.append({'card':card,'protocol':protocol,'repeat':repeat,'model':name,
                            'seed':seed,'true':CLASSES[k],'predicted':CLASSES[j],'count':int(cm[k,j])})
                for k in range(7):
                    recalls.append({'card':card,'protocol':protocol,'repeat':repeat,'model':name,
                        'seed':seed,'class':CLASSES[k],'support':int(cm[k].sum()),
                        'recall':float(cm[k,k]/cm[k].sum()) if cm[k].sum() else np.nan})
                for item in h:
                    histories.append({'card':card,'protocol':protocol,'repeat':repeat,**item})
                print(card,protocol,repeat,name,seed,'BAcc',round(row['bacc'],3),
                      '| sec',round(elapsed,1))

scores = pd.DataFrame(scores)
predictions = pd.DataFrame(predictions)
for filename,frame in [('scores.csv',scores),('test_predictions.csv',predictions),
                       ('class_recall.csv',pd.DataFrame(recalls)),('confusion_counts.csv',pd.DataFrame(cms)),
                       ('training_history.csv',pd.DataFrame(histories))]:
    frame.to_csv(OUT/filename,index=False)
assert len(scores)==56 and len(predictions)==56*257, 'Incomplete E4 model/split grid'
print('Saved:',OUT)

## 4. Summary table

Report balanced accuracy, macro-F1, and per-class recalls for the future-day evaluation and the separate random-split diagnostic.

In [ ]:
agg = (scores.groupby(['model','card','protocol'],as_index=False)
        .agg(bacc=('bacc','mean'),sd=('bacc','std'),f1=('macro_f1','mean'),
             seconds=('fit_seconds','mean'),runs=('seed','size')))
wide = agg.pivot(index='model',columns=['card','protocol'],values='bacc').reindex(METHODS)
wide.to_csv(OUT/'table_E4.csv')
print('Balanced accuracy (future & random):')
display((100*wide).round(1))

# An intentionally compact table for the ACM manuscript.
def fmt(model,card,protocol):
    q=agg[(agg.model==model)&(agg.card==card)&(agg.protocol==protocol)].iloc[0]
    v=f'{100*q.bacc:.1f}'
    return v if q.runs==1 else v+r' $\pm$ '+f'{100*q.sd:.1f}'
lines=[r'\begin{table*}[t]',r'\centering\small',
       r'\caption{E4: model-family comparison on paired single-gas campaigns. '
       r'Balanced accuracy (\%) on matched random splits and the subsequent-day test. '
       r'Dispersion reflects runs or resamples, not independent days.}',
       r'\label{tab:e4_models}',r'\begin{tabular}{@{}lrrrr@{}}',r'\toprule',
       r'Model & 171 random & 171 future & 172 random & 172 future \\',r'\midrule']
for name in METHODS:
    lines.append(name.replace('_',r'\_')+' & '+' & '.join(fmt(name,c,p)
           for c,p in ((171,'random'),(171,'future'),(172,'random'),(172,'future')))+r' \\')
lines.extend([r'\bottomrule',r'\end{tabular}',r'\end{table*}'])
(OUT/'E4_table.tex').write_text('\n'.join(lines)+'\n',encoding='utf-8')

settings={'archives':ARCHIVES,'classes':CLASSES,'methods':METHODS,
          'future_split':(244,134,257),'random_split_seeds':RANDOM_SPLITS,
          'chrono_network_seeds':CHRONO_SEEDS,'random_network_seed':RANDOM_DEEP_SEED,
          'epochs_cap':EPOCHS,'patience':PATIENCE,'device':str(DEVICE),
          'versions':{'torch':torch.__version__,'numpy':np.__version__,
                      'sklearn':__import__('sklearn').__version__},
          'esdata_features_sha256':hashlib.sha256((SRC/'esdata/transforms/features.py').read_bytes()).hexdigest(),
          'archives_meta':{str(c):{'bytes':(DATA_DIR/a).stat().st_size,'mtime_ns':(DATA_DIR/a).stat().st_mtime_ns} for c,a in ARCHIVES.items()}}
(OUT/'config.json').write_text(json.dumps(settings,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
# A small, manuscript-ready visualization of future-day performance.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8.3,3.5))
xx=np.arange(len(METHODS)); w=.36
for shift,card in ((-w/2,171),(w/2,172)):
    vals=[100*agg[(agg.model==m)&(agg.card==card)&(agg.protocol=='future')].iloc[0].bacc for m in METHODS]
    ax.bar(xx+shift,vals,width=w,label=f'Card {card}')
ax.set_xticks(xx,METHODS,rotation=25,ha='right')
ax.set_ylim(0,100);ax.set_ylabel('Balanced accuracy (%)');ax.legend(frameon=False)
fig.tight_layout();fig.savefig(OUT/'fig_E4_future.pdf');plt.close(fig)
print('For the paper:',OUT/'E4_table.tex',OUT/'fig_E4_future.pdf')

### Interpretation

A random-to-chronological difference reflects the split protocol and recorded conditions, not an isolated causal estimate of sensor aging.